In [ ]:
import json, datasets
from pathlib import Path
import pandas as pd
from collections import OrderedDict

regression_file = '/home/hieunt/verl/regression_data/test.json'
path = Path(regression_file)
with path.open('r', encoding='utf-8') as f:
    data = json.load(f, object_pairs_hook=OrderedDict)
    
keys_in_order = list(data.keys())
total_epochs = max((v.get('total_epochs_seen', 0) for v in data.values()), default=0)

rows = []
for epoch_idx in range(total_epochs):
    for qid in keys_in_order:
        rec = data[qid]
        mean_list = rec.get('mean_acc_per_epoch', [])
        std_list = rec.get('std_acc_per_epoch', [])
        rows.append({
            'epoch': epoch_idx + 1,
            'question_index': qid,
            'mean_acc_per_epoch': mean_list[epoch_idx] if epoch_idx < len(mean_list) else None,
            'std_acc_per_epoch': std_list[epoch_idx] if epoch_idx < len(std_list) else None,
        })
time_df = pd.DataFrame(rows)

# print(f"Epochs: {total_epochs}, Questions: {len(keys_in_order)}, Rows: {len(time_df)}")
# try:
#     display(time_df.head(30))
#     # display(time_df.tail(20))
# except NameError:
#     print(time_df.head(10).to_string(index=False))

dapo_file_path = '/home/hieunt/verl/data/dapo-math-17k.parquet'
parquet_df = datasets.load_dataset('parquet', data_files=[dapo_file_path])['train']
# parquet_df['prompt'][0][0]['content']
# 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.\n\nRemember to put your answer on its own line after "Answer:".'
# parquet_df['extra_info'][0]['index]
# 9a9b6eb4-a1cb-49d1-8c1e-62eaf2f74079



In [31]:
# Map question_index to the first prompt message's content from parquet_df
# Uses extra_info['index'] as the key
id_to_prompt_content = {}
for prompt_messages, extra in zip(parquet_df['prompt'], parquet_df['extra_info']):
    idx_value = None
    if isinstance(extra, dict):
        idx_value = extra.get('index')
    else:
        try:
            idx_value = extra['index']
        except Exception:
            idx_value = None

    first_message_content = None
    if isinstance(prompt_messages, list) and len(prompt_messages) > 0:
        first_message = prompt_messages[0]
        if isinstance(first_message, dict):
            first_message_content = first_message.get('content')

    if idx_value is not None:
        id_to_prompt_content[str(idx_value)] = first_message_content

# Create the new column in time_df
time_df['prompt_content'] = time_df['question_index'].astype(str).map(id_to_prompt_content)


In [35]:
time_df[~time_df['mean_acc_per_epoch'].notna()]

,epoch,question_index,mean_acc_per_epoch,std_acc_per_epoch,prompt_content
298,2,b04c44de-3e20-4766-b897-29fe9cbe9101,NaN,NaN,Solve the following math problem step by step....
310,2,43970b7d-8909-4c31-8cde-970e547a5043,NaN,NaN,Solve the following math problem step by step....
413,2,d95a5367-c6ea-4fa3-8562-26294b2a3e5b,NaN,NaN,Solve the following math problem step by step....
